In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import xarray as xr
import matplotlib.pyplot as plt
import contextily as cx
from shapely.geometry import box, mapping, shape, Point

import pystac_client
import planetary_computer as pc
import odc.stac
from rasterio.features import shapes as raster_to_shapes
import folium
from folium.plugins import GroupedLayerControl
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [5]:
catalog= pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

In [11]:
reference_p= Point(-63.90, -8.76)
wsa=reference_p.buffer(0.5)
recent_win = "2023-06-01/2023-09-30"
collect=["sentinel-2-l2a"]
half_size = 0.05  # roughly a 10 km box

In [9]:
# function that finds the scenes of interest 
def search_scene(reference_point,wide_search_area,recent_window,c,coll):
    candidates_nearby = c.search(
    collections=coll,
    intersects=mapping(wide_search_area),
    datetime=recent_window,
    query={"eo:cloud_cover": {"lt": 30}},
    ).items()
    candidates_nearby = list(candidates_nearby)
    return candidates_nearby


In [ ]:
candidates_near= search_scene(reference_p,wsa,recent_win, catalog,collect)

In [12]:
# pick the scene whose real footprint covers the most area, usually the one furthest from a tile edge
def footprint (candidates_n, half_s): 

    reference_scene = max(candidates_n, key=lambda item: shape(item.geometry).area)
    footprint = shape(reference_scene.geometry)
    inner_point = footprint.buffer(-0.15).representative_point()
    bbox = [inner_point.x - half_s, inner_point.y - half_s,
            inner_point.x + half_s, inner_point.y + half_s]
    aoi = box(*bbox)
    aoi_geojson = mapping(aoi)
    return bbox,aoi_geojson

bbox,aoi_geojson = footprint (candidates_near, half_size)

In [13]:
bbox

[-63.461313123204675,
 -9.592136744908483,
 -63.36131312320468,
 -9.492136744908482]